# 🚗 Understanding PID Control
## A Hands-On Guide for the Never-Crash Vehicle

---

### How to Use This Notebook

| Label | Who | What |
|-------|-----|------|
| 🎯 **DEMO** | Instructor runs | Watch, listen, ask questions |
| 🤝 **TOGETHER** | We work as a class | Shout out predictions! |
| ✏️ **PRACTICE** | You alone | Fill in the `...` to make it work |

---

### The Challenge

Your vehicle needs to maintain exactly **10 cm** from a wall while cruising.

- Too far? → Drive closer
- Too close? → Back up
- Just right? → Hold it there *perfectly*

Simple to say. Hard to do. By the end of this notebook you'll understand exactly
why we need PID — and what each of P, I, and D contributes.

---

### The Formula We're Building

```
throttle = Kp × error  +  Ki × ∫error dt  +  Kd × (Δerror/Δt)
            ────────────    ──────────────      ────────────────
               P term           I term               D term
          "React to NOW"   "Remember PAST"    "Predict FUTURE"
```

where **error = current_distance − target_distance**

Let's build this up one piece at a time.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Consistent style throughout ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize'  : [13, 5],
    'font.size'       : 11,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'lines.linewidth' : 2.2,
})

# ── Colors we'll use for each PID term ───────────────────────────────────────
C_P      = '#FF8C00'   # orange  — Proportional
C_I      = '#2196F3'   # blue    — Integral
C_D      = '#9C27B0'   # purple  — Derivative
C_PID    = '#2E7D32'   # dark green — full PID
C_target = '#2E7D32'   # same green for target line
C_dist   = '#1565C0'   # dark blue — distance line
C_err    = '#E53935'   # red — error

TARGET = 10.0          # cm — same as your vehicle code

print('Setup complete. TARGET =', TARGET, 'cm')

## The Simulation Engine

The cell below is a **physics simulation** of your car and wall.
We'll use it throughout the notebook so you can *see* what PID does
before you put it on the physical vehicle.

The physics model:
- Car has mass (inertia) — it takes time to speed up or slow down
- Throttle creates force toward the wall; drag opposes motion
- `disturbance` = constant force *away* from wall (like driving up a slope)

You don't need to understand every line — just run it once, then use `simulate(...)`.


In [ ]:
def simulate(Kp=0.0, Ki=0.0, Kd=0.0,
             initial_distance=30.0, target=TARGET,
             dt=0.05, steps=300, disturbance=0.0):
    '''
    Simulate a car maintaining distance from a wall using PID.

    Positive throttle  → car moves toward wall → distance decreases
    disturbance > 0    → constant force pushing car AWAY from wall
    '''
    MAX_ACCEL = 20.0   # cm/s² at full throttle
    DRAG      = 2.0    # velocity damping (friction)

    distances  = [initial_distance]
    errors, P_log, I_log, D_log, throttle_log = [], [], [], [], []

    distance   = initial_distance
    velocity   = 0.0
    integral   = 0.0
    last_error = 0.0

    for step in range(steps):
        error      = distance - target
        integral  += error * dt
        derivative = (error - last_error) / dt if step > 0 else 0.0

        P        = Kp * error
        I        = Ki * integral
        D        = Kd * derivative
        throttle = max(-1.0, min(1.0, P + I + D))

        # Physics
        net_force = throttle * MAX_ACCEL - DRAG * velocity - disturbance
        velocity += net_force * dt
        distance  = max(0.5, distance - velocity * dt)

        errors.append(error); P_log.append(P); I_log.append(I)
        D_log.append(D);      throttle_log.append(throttle)
        distances.append(distance)
        last_error = error

    t = np.linspace(0, steps * dt, steps)
    return dict(time=t,
                distance=np.array(distances[:steps]),
                error=np.array(errors),
                P=np.array(P_log), I=np.array(I_log), D=np.array(D_log),
                throttle=np.array(throttle_log))

print('Simulation engine loaded ✓')
print('Usage: data = simulate(Kp=0.08, Ki=0.015, Kd=0.5)')

---
## Part 1 — Why Simple Control Fails

Before we learn PID, let's see what happens with *simpler* strategies.
Two common approaches:

1. **Open loop** — set throttle to a constant, ignore the sensor
2. **Bang-bang** — "if too far, go full speed; if too close, full reverse"

🎯 **DEMO** — Run the cell below and we'll discuss what goes wrong with each.


In [ ]:
# 🎯 DEMO: Two naive strategies — and why they fail

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
t = np.linspace(0, 5, 200)

# ── Left: open loop (constant throttle, no sensor feedback) ──────────────────
for speed, color, lbl in [(4, '#90CAF9', 'Slow (4 cm/s)'),
                           (10, '#FF9800', 'Medium (10 cm/s)'),
                           (20, '#F44336', 'Fast (20 cm/s)')]:
    dist = np.maximum(30 - speed * t, 0)
    axes[0].plot(t, dist, color=color, label=lbl)

axes[0].axhline(TARGET, color=C_target, linestyle='--', lw=2, label=f'Target ({TARGET} cm)')
axes[0].fill_between([0,5], [-1,-1], [0,0], color='gray', alpha=0.4)
axes[0].text(2.5, -0.7, '🧱 WALL', ha='center', fontsize=9, fontweight='bold')
axes[0].set(xlim=[0,5], ylim=[-1.5, 33], xlabel='Time (s)', ylabel='Distance (cm)',
            title='Open Loop — constant throttle\n"Just drive and hope for the best"')
axes[0].legend(fontsize=9)

# ── Right: bang-bang (on/off) ─────────────────────────────────────────────────
dist_bb = 30.0
dists_bb = [dist_bb]
for _ in range(199):
    spd = 14 if dist_bb > TARGET else -14
    dist_bb = max(0, dist_bb - spd * 0.025)
    dists_bb.append(dist_bb)

axes[1].plot(t, dists_bb[:200], color='#9C27B0', lw=2)
axes[1].axhline(TARGET, color=C_target, linestyle='--', lw=2, label=f'Target ({TARGET} cm)')
axes[1].set(xlim=[0,5], ylim=[-1.5, 33], xlabel='Time (s)', ylabel='Distance (cm)',
            title='Bang-Bang — full gas or full brake\n"React as hard as possible"')
axes[1].legend()

plt.suptitle('Neither strategy works well — we need something smarter', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Open loop:  ignores the sensor — crashes at any speed!')
print('Bang-bang:  sensor IS used, but overcorrects — oscillates forever')
print()
print('What we need: a response that is PROPORTIONAL to how wrong we are.')

---
## Part 2 — The Error Signal

Every PID controller is built on one idea:

> **error = current_distance − target_distance**

- `error > 0` → car is too far away → need to move forward (positive throttle)
- `error < 0` → car is too close → need to back up (negative throttle)
- `error = 0` → perfect! 🎯

This single number is the *language* the controller speaks.
Let's visualize it.


In [ ]:
# 🎯 DEMO: The error signal

data = simulate(Kp=0.08, steps=200)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

# Distance
ax1.plot(data['time'], data['distance'], color=C_dist, label='Actual distance')
ax1.axhline(TARGET, color=C_target, linestyle='--', lw=2, label=f'Target = {TARGET} cm')
ax1.fill_between(data['time'], data['distance'], TARGET,
                 where=data['distance'] > TARGET, alpha=0.15, color='orange', label='Too far (error > 0)')
ax1.fill_between(data['time'], data['distance'], TARGET,
                 where=data['distance'] < TARGET, alpha=0.15, color='red',    label='Too close (error < 0)')
ax1.set(ylabel='Distance (cm)', title='Car Distance vs Target', ylim=[0, 35])
ax1.legend(loc='upper right', fontsize=9)

# Annotate error at one moment
idx = 30
ax1.annotate(f'error = {data["distance"][idx]:.1f} − {TARGET} = {data["error"][idx]:.1f} cm',
             xy=(data['time'][idx], data['distance'][idx]),
             xytext=(data['time'][idx]+1.5, data['distance'][idx]+5),
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

# Error
ax2.plot(data['time'], data['error'], color=C_err, label='error = distance − target')
ax2.axhline(0, color=C_target, linestyle='--', lw=2, label='error = 0  (perfect!)')
ax2.fill_between(data['time'], data['error'], 0,
                 where=data['error'] > 0, alpha=0.2, color='orange')
ax2.fill_between(data['time'], data['error'], 0,
                 where=data['error'] < 0, alpha=0.2, color='red')
ax2.set(xlabel='Time (seconds)', ylabel='Error (cm)', title='Error = Distance − Target')
ax2.legend()

plt.tight_layout()
plt.show()

print('KEY: error = current_distance - TARGET_DISTANCE')
print('  Positive → too far   → go forward')
print('  Negative → too close → back up')
print('  Zero     → perfect!')

---
## Part 3 — Proportional Control (P)

The simplest idea: **make the throttle proportional to the error**.

```python
error    = current_distance - TARGET_DISTANCE
P        = Kp * error
throttle = P
```

- Big error → big throttle (move fast when far away)
- Small error → small throttle (ease in near the target)
- Zero error → zero throttle (stop exactly at target... in theory)

`Kp` is a **gain** — a dial that controls how aggressively the car reacts.

🎯 **DEMO** — Run the cell to see how Kp affects behavior.


In [ ]:
# 🎯 DEMO: Three different Kp values

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

configs = [
    (0.02, '#90CAF9', 'Kp = 0.02  (Too Small)\nSluggish — barely reacts'),
    (0.08, '#4CAF50', 'Kp = 0.08  (Good)\nDecent response'),
    (0.28, '#F44336', 'Kp = 0.28  (Too Large)\nOscillates — overcorrects'),
]

for ax, (kp, color, title) in zip(axes, configs):
    data = simulate(Kp=kp, steps=300)
    ax.plot(data['time'], data['distance'], color=color, lw=2.5)
    ax.axhline(TARGET, color=C_target, linestyle='--', lw=2)
    ax.fill_between(data['time'], TARGET-1, TARGET+1, alpha=0.1, color='green',
                    label='±1 cm window')
    ax.set(xlabel='Time (s)', title=title, ylim=[0, 35])

    settled = np.where(np.abs(data['error']) < 1.0)[0]
    if len(settled):
        ts = data['time'][settled[0]]
        ax.axvline(ts, color=color, linestyle=':', alpha=0.8)
        ax.text(ts+0.2, 27, f'settles\n~{ts:.0f}s', color=color, fontsize=9)

axes[0].set_ylabel('Distance (cm)')
plt.suptitle('P-Only Control:   throttle = Kp × (distance − target)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Observation:')
print('  Small Kp → slow approach, may never quite reach target')
print('  Large Kp → fast but oscillates, overshoots, may be unstable')
print('  "Just right" → depends on the physical system!')

### 🤝 TOGETHER — Find the Best Kp

Change `KP_VALUE` below and run the cell.
Before you run: **predict** what will happen. Will it be faster? Slower? More oscillation?


In [ ]:
# 🤝 TOGETHER: Tune Kp — change this and predict before running!

KP_VALUE = 0.08   # ← Try values like 0.01, 0.05, 0.15, 0.30, 0.50

data = simulate(Kp=KP_VALUE, steps=300)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax1.plot(data['time'], data['distance'], color=C_P, lw=2.5)
ax1.axhline(TARGET, color=C_target, linestyle='--', lw=2)
ax1.fill_between(data['time'], TARGET-1, TARGET+1, alpha=0.12, color='green')
ax1.set(ylabel='Distance (cm)', title=f'P-Only  Kp={KP_VALUE}', ylim=[0, 35])

ax2.plot(data['time'], data['throttle'], color='steelblue', lw=2)
ax2.axhline(0, color='k', lw=0.8)
ax2.set(xlabel='Time (s)', ylabel='Throttle', ylim=[-1.2, 1.2])

plt.tight_layout()
plt.show()

settled = np.where(np.abs(data['error']) < 0.5)[0]
settle_str = f'{data["time"][settled[0]]:.1f}s' if len(settled) else 'never (within 15s)'
print(f'Kp={KP_VALUE}  |  Settles within 0.5 cm at: {settle_str}')
print(f'Final error: {data["error"][-1]:.2f} cm')

### ✏️ PRACTICE — Implement Proportional Control

Fill in the two `...` lines below.
This is exactly the code that goes in **vehicle-2.py**.


In [ ]:
# ✏️ PRACTICE: Implement P

def p_controller(current_distance, target, Kp):
    # Step 1: calculate the error
    error = ...                          # YOUR CODE

    # Step 2: proportional term
    P = ...                              # YOUR CODE

    # Step 3: clamp to motor limits
    throttle = max(-1.0, min(P, 1.0))
    return throttle

# ── Test it ──────────────────────────────────────────────────────────────────
print('Testing your P controller:')
tests = [(30, 10, 0.08), (10, 10, 0.08), (5, 10, 0.08)]
for dist, tgt, kp in tests:
    try:
        result = p_controller(dist, tgt, kp)
        ok = (result > 0 and dist > tgt) or (result < 0 and dist < tgt) or (result == 0 and dist == tgt)
        sign = 'pos ✓' if result > 0 else ('neg ✓' if result < 0 else 'zero ✓')
        print(f'  dist={dist:5.1f}, target={tgt} → throttle={result:+.4f}  ({sign if ok else "WRONG ✗"})')
    except Exception as e:
        print(f'  Error: {e}')

---
## Part 4 — The Problem with P Alone

P control has **two failure modes**:

### 1. Oscillation (Kp too high)
Car overshoots the target, then overcorrects, then overshoots again...

### 2. Steady-State Error (real-world disturbances)
Imagine your car is driving on a slight slope, or fighting a headwind.
Some constant force is always pushing the car away from the wall.

With P-only, the car can *only* apply throttle when there's an error.
So it reaches an equilibrium **slightly off-target** where:
> `Kp × error_ss = disturbance_force`

The car will never reach exactly 10 cm — it settles at 10 + error_ss.

🎯 **DEMO** — Let's see this steady-state error in action.


In [ ]:
# 🎯 DEMO: Steady-state error — P alone can't fight a constant disturbance

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: P-only WITH disturbance
data_p = simulate(Kp=0.08, disturbance=3.0, steps=400)
ss_dist = np.mean(data_p['distance'][-60:])
ss_err  = ss_dist - TARGET

axes[0].plot(data_p['time'], data_p['distance'], color=C_P, lw=2.5, label='P-only + disturbance')
axes[0].axhline(TARGET,  color=C_target, linestyle='--', lw=2, label=f'Target ({TARGET} cm)')
axes[0].axhline(ss_dist, color=C_err,    linestyle=':',  lw=2,
                label=f'Settles at {ss_dist:.1f} cm (error = {ss_err:.1f} cm)')
axes[0].fill_between(data_p['time'], TARGET, ss_dist, alpha=0.15, color='red')
axes[0].set(xlabel='Time (s)', ylabel='Distance (cm)',
            title='P-Only with disturbance\n(steady-state error — never reaches target)', ylim=[5,35])
axes[0].legend(fontsize=9)

# Right: explain the math
axes[1].axis('off')
txt = (
    'Why does this happen?\n\n'
    'The car settles when throttle\n'
    'exactly balances the disturbance:\n\n'
    '   P = disturbance_force\n'
    '   Kp x error_ss = disturbance\n\n'
    f'   {0.08} x error_ss = 3.0\n'
    f'   error_ss = {3.0/0.08:.2f} cm\n\n'
    'Car stops at 10 + 1.88 = 11.88 cm\n\n'
    'To fix this, we need a term\n'
    'that KEEPS PUSHING even when\n'
    'the error is small but persistent.\n\n'
    '->  Enter: the Integral term'
)
axes[1].text(0.1, 0.95, txt, transform=axes[1].transAxes,
             va='top', fontsize=11, fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.8))
axes[1].set_title('The Math Behind Steady-State Error', fontsize=11)

plt.suptitle('P Alone Cannot Eliminate a Constant Disturbance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 5 — Integral: The Memory of Past Mistakes

> **"How long have I been off-target?"**

The integral accumulates error over time:

```python
integral += error * dt        # add a tiny slice of error each timestep
I = Ki * integral             # integral term
```

**The key insight:** Even a tiny, persistent error adds up.

- At each timestep, you add a small slice: `error × dt`
- This is literally the **area under the error curve**
- After many timesteps, the integral becomes large enough to *overcome* the disturbance

Once the car reaches the target (error = 0), the integral **holds** its value —
maintaining exactly the throttle needed to fight the disturbance. This is called
**integral memory**.

🎯 **DEMO** — Watch the area accumulate below.


In [ ]:
# 🎯 DEMO: The Integral — area under the error curve

data = simulate(Kp=0.08, disturbance=3.0, steps=400)
t, err = data['time'], data['error']

fig = plt.figure(figsize=(13, 9))
gs  = GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.35)

ax_dist = fig.add_subplot(gs[0, :])
ax_err  = fig.add_subplot(gs[1, :])
ax_int  = fig.add_subplot(gs[2, 0])
ax_txt  = fig.add_subplot(gs[2, 1])

# ── Distance ──────────────────────────────────────────────────────────────────
ss = np.mean(data['distance'][-60:])
ax_dist.plot(t, data['distance'], color=C_dist, lw=2.5)
ax_dist.axhline(TARGET, color=C_target, linestyle='--', lw=2, label=f'Target')
ax_dist.axhline(ss, color=C_err, linestyle=':', lw=2, label=f'Settles at {ss:.1f} cm')
ax_dist.fill_between(t, TARGET, data['distance'],
                     where=data['distance'] > TARGET, alpha=0.15, color='red')
ax_dist.set(ylabel='Distance (cm)', title='P-Only with Disturbance: Persistent error',
            ylim=[5,35])
ax_dist.legend(fontsize=9)

# ── Error + shaded integral ───────────────────────────────────────────────────
ax_err.plot(t, err, color=C_err, lw=2.5)
ax_err.fill_between(t, 0, err, where=err>=0, alpha=0.30, color='orange',
                    label='Accumulated area = integral')
ax_err.fill_between(t, 0, err, where=err<0,  alpha=0.30, color='steelblue',
                    label='Negative area (subtracts)')
ax_err.axhline(0, color=C_target, linestyle='--', lw=2)

# Highlight one timestep slice
idx = 120
ax_err.axvspan(t[idx], t[idx]+0.1, alpha=0.7, color='red', zorder=5)
ax_err.annotate(f'Slice = error × dt\n= {err[idx]:.2f} × 0.05',
                xy=(t[idx]+0.05, err[idx]/2),
                xytext=(t[idx]+2.5, err[idx]+1.5),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                fontsize=9, color='red')
ax_err.set(ylabel='Error (cm)', title='Error over time — shaded area = integral accumulating')
ax_err.legend(fontsize=9)

# ── Integral value ────────────────────────────────────────────────────────────
integral_vals = np.cumsum(err) * 0.05
ax_int.plot(t, integral_vals, color=C_I, lw=2.5)
ax_int.set(xlabel='Time (s)', ylabel='Integral (cm·s)',
           title='Integral value keeps growing\nwhile error persists')
ax_int.fill_between(t, 0, integral_vals, alpha=0.2, color=C_I)

# ── Text explanation ──────────────────────────────────────────────────────────
ax_txt.axis('off')
final_int = integral_vals[-1]
Ki_demo   = 0.015
I_effect  = Ki_demo * final_int
explanation = (
    'After the car settles:\n\n'
    f'  Error ~ {np.mean(err[-60:]):.2f} cm  (small but persistent)\n'
    f'  Integral ~ {final_int:.1f} cm*s  (LARGE from accumulating)\n\n'
    f'  I = Ki x integral\n'
    f'  I = {Ki_demo} x {final_int:.0f} = {I_effect:.2f}\n\n'
    'This extra throttle overcomes\n'
    'the disturbance, driving error\n'
    'all the way to zero.\n\n'
    '"I have been off-target for a\n'
    'long time - I need to push harder."'
)
ax_txt.text(0.05, 0.95, explanation, transform=ax_txt.transAxes,
            va='top', fontsize=10, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.9))
ax_txt.set_title('Why the Integral Wins', fontsize=11)

plt.suptitle('THE INTEGRAL: Accumulated Error Over Time  (area under the curve)',
             fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# 🎯 DEMO: PI eliminates steady-state error

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (ki, color, title) in zip(axes, [
    (0.0,  C_P,   'P-Only  (Kp=0.08, Ki=0)\nStuck with steady-state error'),
    (0.015, C_I,  'PI Control  (Kp=0.08, Ki=0.015)\nFinds the target!')
]):
    data = simulate(Kp=0.08, Ki=ki, disturbance=3.0, steps=400)
    ax.plot(data['time'], data['distance'], color=color, lw=2.5)
    ax.axhline(TARGET, color=C_target, linestyle='--', lw=2, label=f'Target {TARGET} cm')
    ax.fill_between(data['time'], TARGET-0.5, TARGET+0.5,
                    alpha=0.15, color='green', label='±0.5 cm window')
    final = abs(data['error'][-1])
    ax.set(xlabel='Time (s)', title=title, ylim=[4, 35])
    ax.text(0.02, 0.06, f'Final error: {final:.2f} cm',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.legend(fontsize=9)

axes[0].set_ylabel('Distance (cm)')
plt.suptitle('Adding Integral (I) Eliminates Steady-State Error', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### ✏️ PRACTICE — Implement the Integral Term

Three lines to fill in this time. Think about the formula:

```
integral += error * dt        (accumulate a slice each timestep)
I = Ki * integral             (take a fraction of the total)
```


In [ ]:
# ✏️ PRACTICE: Implement the Integral

def pi_controller(current_distance, target, Kp, Ki,
                  integral, dt):
    '''
    Returns (throttle, updated_integral).
    integral and dt are passed in from the calling loop.
    '''
    # Step 1: error
    error = current_distance - target

    # Step 2: Proportional
    P = Kp * error

    # Step 3: Integral — accumulate the error over time
    integral += ...            # YOUR CODE  (hint: error × dt)
    I = ...                    # YOUR CODE  (hint: Ki × integral)

    throttle = max(-1.0, min(P + I, 1.0))
    return throttle, integral

# ── Test ──────────────────────────────────────────────────────────────────────
print('Testing Integral implementation:')
integ = 0.0
dist_seq = [30, 25, 20, 16, 13, 11, 10.5, 10.2, 10.05]
for d in dist_seq:
    try:
        thr, integ = pi_controller(d, TARGET, 0.08, 0.015, integ, 0.05)
        print(f'  dist={d:5.2f}  integral={integ:7.3f}  throttle={thr:+.4f}')
    except Exception as e:
        print(f'  Error: {e}')
        break

---
## Part 6 — Derivative: Anticipating the Future

> **"How fast is the error changing?"**

The derivative measures the *rate of change* of the error — the slope of the error curve:

```python
derivative = (error - last_error) / dt
D = Kd * derivative
```

**The key insight:** If the car is approaching fast, the error is dropping quickly
(large negative derivative). The D term adds a *braking* force to prevent overshoot.

- Steep negative slope → approaching quickly → D reduces throttle (slow down!)
- Steep positive slope → moving away quickly → D increases throttle (resist it!)
- Flat slope → error not changing → D ≈ 0

This is why D is called the **predictive** term — it reacts to *how things are changing*,
not just what the error is right now.

🎯 **DEMO** — Let's see the derivative as slope on the error curve.


In [ ]:
# 🎯 DEMO: Derivative = slope of the error curve

t   = np.linspace(0, 6, 600)
dt  = t[1] - t[0]
err = 18 * np.exp(-0.45 * t) * np.cos(2.2 * t + 0.4)   # rich oscillating signal

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(t, err, color=C_dist, lw=2.5, label='Error over time', zorder=5)
ax1.axhline(0, color=C_target, linestyle='--', lw=2, label='Error = 0  (perfect!)')
ax1.set(ylabel='Error (cm)', title='Derivative = Slope at Each Moment')

moments = [
    (0.25, '#E53935', 'Falling steeply\n(approaching fast)\nD brakes hard'),
    (1.45, '#FF9800', 'Near zero crossing\n(error small, changing fast)'),
    (2.50, '#9C27B0', 'Rising steeply\n(moving away fast)\nD pushes back'),
    (4.10, '#4CAF50', 'Nearly flat\n(almost settled)\nD ≈ 0'),
]

for t_c, color, label in moments:
    idx = int(t_c / dt)
    idx = max(1, min(idx, len(t)-2))
    slope = (err[min(idx+1, len(err)-1)] - err[max(idx-1,0)]) / (2*dt)
    t_tang = np.array([t_c-0.55, t_c+0.55])
    e_tang = err[idx] + slope * (t_tang - t_c)
    ax1.plot(t_tang, e_tang, color=color, lw=2, linestyle='--', alpha=0.9)
    ax1.plot(t_c, err[idx], 'o', color=color, ms=11, zorder=10)
    ax1.annotate(f'slope={slope:.1f}\n{label}',
                 xy=(t_c, err[idx]),
                 xytext=(t_c+0.35, err[idx]+(5 if err[idx]<0 else -5)),
                 fontsize=8, color=color,
                 arrowprops=dict(arrowstyle='->', color=color, lw=1.2))
ax1.legend(fontsize=9)

# Derivative over time
deriv = np.gradient(err, t)
ax2.plot(t, deriv, color=C_D, lw=2.5, label='Derivative (slope of error)')
ax2.fill_between(t, 0, deriv, where=deriv<0, alpha=0.2, color=C_I,
                 label='Negative: error falling (approaching)')
ax2.fill_between(t, 0, deriv, where=deriv>0, alpha=0.2, color=C_err,
                 label='Positive: error rising (moving away)')
ax2.axhline(0, color=C_target, linestyle='--', lw=2)
ax2.set(xlabel='Time (s)', ylabel='d(error)/dt  (cm/s)',
        title='Derivative Term — D = Kd × derivative')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print()
print('Large NEGATIVE derivative → car is closing in fast → D REDUCES throttle → prevents crash')
print('Large POSITIVE derivative → car is moving away fast → D INCREASES throttle → fights it')
print('Near-zero derivative      → error not changing much → D contributes little')

In [ ]:
# 🎯 DEMO: D reduces oscillation

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (kd, color, title) in zip(axes, [
    (0.0,  C_P,  'P-Only  (Kp=0.25, Kd=0)\nOscillates — keeps overshooting'),
    (1.2,  C_D,  'PD Control  (Kp=0.25, Kd=1.2)\nDamped — much smoother!'),
]):
    data = simulate(Kp=0.25, Kd=kd, steps=300)
    ax.plot(data['time'], data['distance'], color=color, lw=2.5)
    ax.axhline(TARGET, color=C_target, linestyle='--', lw=2)
    ax.fill_between(data['time'], TARGET-1, TARGET+1, alpha=0.1, color='green')
    ax.set(xlabel='Time (s)', title=title, ylim=[0, 35])

axes[0].set_ylabel('Distance (cm)')
plt.suptitle('Adding Derivative (D) Damps Oscillation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Why does D damp oscillation?')
print('  As car zooms past target, error changes fast (big negative derivative)')
print('  D term fires a strong REVERSE signal → slows the car before it overshoots')
print('  Think of D as: the faster you are heading somewhere wrong, the harder you brake')

### 🤝 TOGETHER — Explore the Derivative

Change `KD_VALUE` below. Try values between 0 and 3.0.
**Predict first:** What happens if Kd is very large?


In [ ]:
# 🤝 TOGETHER: Explore Kd (start with Kp=0.20 so oscillation is visible)

KD_VALUE = 0.0   # ← Try 0, 0.3, 0.8, 1.5, 3.0, 6.0

data = simulate(Kp=0.20, Kd=KD_VALUE, steps=300)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax1.plot(data['time'], data['distance'], color=C_D, lw=2.5)
ax1.axhline(TARGET, color=C_target, linestyle='--', lw=2)
ax1.fill_between(data['time'], TARGET-1, TARGET+1, alpha=0.12, color='green')
ax1.set(ylabel='Distance (cm)', title=f'PD Control  Kp=0.20  Kd={KD_VALUE}', ylim=[0, 35])

ax2.plot(data['time'], data['D'], color=C_D, lw=2, label=f'D term (Kd={KD_VALUE})')
ax2.plot(data['time'], data['P'], color=C_P, lw=2, label='P term', alpha=0.7)
ax2.axhline(0, color='k', lw=0.8)
ax2.set(xlabel='Time (s)', ylabel='Term value', ylim=[-1.2, 1.2])
ax2.legend()

plt.tight_layout()
plt.show()

### ✏️ PRACTICE — Implement the Derivative Term

```
derivative = (current_error - last_error) / dt
D = Kd * derivative
```

`last_error` is what the error was in the *previous loop iteration*. That's why
vehicle-2.py keeps a global variable `last_error` — to remember it between calls.


In [ ]:
# ✏️ PRACTICE: Implement the Derivative

def pid_controller(current_distance, target, Kp, Ki, Kd,
                   last_error, integral, dt):
    '''
    Full PID. Returns (throttle, error, integral).
    Caller stores error as last_error for the next call.
    '''
    error = current_distance - target
    P     = Kp * error

    integral += error * dt
    I         = Ki * integral

    # ── YOUR CODE: derivative and D term ──────────────────────────────────────
    derivative = ...       # (current error - last error) / dt
    D          = ...       # Kd × derivative

    throttle = max(-1.0, min(P + I + D, 1.0))
    return throttle, error, integral

# ── Test ──────────────────────────────────────────────────────────────────────
print('Testing full PID controller:')
last_err = 0.0
integ    = 0.0
for dist in [30, 24, 18, 14, 12, 10.8, 10.2, 10.05]:
    try:
        thr, last_err, integ = pid_controller(
            dist, TARGET, 0.08, 0.015, 0.5, last_err, integ, 0.05)
        print(f'  dist={dist:5.2f}  P+I+D → throttle={thr:+.4f}')
    except Exception as e:
        print(f'  Error: {e}'); break

---
## Part 7 — Full PID in Action

Now we combine all three terms:

```
throttle = Kp × error  +  Ki × integral  +  Kd × derivative
```

Each term handles a different failure mode:

| Term | Question it answers | Fixes |
|------|---------------------|-------|
| **P** | "How wrong am I *right now*?" | No response at all |
| **I** | "How long have I *been* wrong?" | Steady-state error |
| **D** | "How *fast* is the error changing?" | Oscillation / overshoot |

🎯 **DEMO** — First, let's see each term's contribution to the throttle signal.
Then the grand comparison: P vs PI vs PD vs PID.


In [ ]:
# 🎯 DEMO: How P, I, and D each contribute to the throttle signal

data = simulate(Kp=0.08, Ki=0.015, Kd=0.5, disturbance=3.0, steps=400)
t    = data['time']

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

# ── Distance ──────────────────────────────────────────────────────────────────
axes[0].plot(t, data['distance'], color=C_PID, lw=2.5, label='Distance (PID)')
axes[0].axhline(TARGET, color=C_target, linestyle='--', lw=2, label='Target')
axes[0].set(ylabel='Distance (cm)', title='Full PID: Distance reaches target and holds', ylim=[4, 35])
axes[0].legend()

# ── Throttle decomposed ───────────────────────────────────────────────────────
p, i_v, d_v = data['P'], data['I'], data['D']
axes[1].plot(t, data['throttle'], 'k', lw=2.0, label='Total throttle', zorder=10)
axes[1].fill_between(t, 0, p,         alpha=0.4, color=C_P, label='P term')
axes[1].fill_between(t, p, p+i_v,     alpha=0.4, color=C_I, label='I term (stacked)')
axes[1].fill_between(t, p+i_v, data['throttle'],
                               alpha=0.4, color=C_D, label='D term (stacked)')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set(ylabel='Throttle', title='P + I + D contributions to total throttle', ylim=[-1.4, 1.4])
axes[1].legend(ncol=4, fontsize=9)

# ── Each term separately ──────────────────────────────────────────────────────
axes[2].plot(t, p,   color=C_P, lw=2, label='P = Kp × error')
axes[2].plot(t, i_v, color=C_I, lw=2, label='I = Ki × integral')
axes[2].plot(t, d_v, color=C_D, lw=2, label='D = Kd × derivative')
axes[2].axhline(0, color='k', lw=0.8)
axes[2].set(xlabel='Time (s)', ylabel='Term value',
            title='Individual P, I, D terms over time')
axes[2].legend(ncol=3)

plt.suptitle('Inside Full PID: Three Terms Working Together', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Notice:')
print('  P  — large at start, shrinks as car approaches, settles near zero')
print('  I  — starts at zero, grows slowly, eventually overcomes the disturbance')
print('  D  — spikes on fast changes, damps oscillation, nearly zero once settled')

In [ ]:
# 🎯 DEMO: The Grand Comparison — P vs PI vs PD vs PID (all with disturbance)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

configs = [
    ('P only',  0.08, 0.000, 0.0, C_P, 'Has steady-state error\n(can\'t fight the disturbance)'),
    ('PI',      0.08, 0.015, 0.0, C_I, 'Fixes steady-state error\nbut may be slower or oscillate'),
    ('PD',      0.08, 0.000, 0.5, C_D, 'Reduces oscillation\nbut still has steady-state error'),
    ('PID',     0.08, 0.015, 0.5, C_PID,'Best of everything:\nfast, stable, no steady-state error'),
]

for ax, (label, kp, ki, kd, color, desc) in zip(axes, configs):
    data = simulate(Kp=kp, Ki=ki, Kd=kd, disturbance=3.0, steps=400)
    ax.plot(data['time'], data['distance'], color=color, lw=2.5)
    ax.axhline(TARGET, color=C_target, linestyle='--', lw=2)
    ax.fill_between(data['time'], TARGET-0.5, TARGET+0.5, alpha=0.12, color='green')
    final_err = abs(data['error'][-1])
    settled = np.where(np.abs(data['error']) < 0.5)[0]
    settle_t = f'{data["time"][settled[0]]:.0f}s' if len(settled) else 'never'
    ax.set(xlabel='Time (s)', ylabel='Distance (cm)', ylim=[4, 35],
           title=f'{label}  (Kp={kp}, Ki={ki}, Kd={kd})\n{desc}')
    ax.text(0.02, 0.08, f'Final error: {final_err:.2f} cm  |  Settles: {settle_t}',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

plt.suptitle('Grand Comparison: P vs PI vs PD vs PID\n(all simulations have same disturbance — constant force away from wall)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 🤝 TOGETHER — Tune Your Own PID

Now you control all three gains. The scoring function will tell you how well you did.

**Goal:** reach the target as fast as possible, with as little overshoot as possible,
and hold it there with minimal steady-state error.


In [ ]:
# 🤝 TOGETHER: Tune your own PID — adjust all three gains

KP = 0.08     # ← Tune this
KI = 0.015    # ← Tune this
KD = 0.5      # ← Tune this

data = simulate(Kp=KP, Ki=KI, Kd=KD, disturbance=3.0, steps=400)
t    = data['time']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(t, data['distance'], color=C_PID, lw=2.5, label='Distance')
ax1.axhline(TARGET, color=C_target, linestyle='--', lw=2, label=f'Target {TARGET} cm')
ax1.fill_between(t, TARGET-0.5, TARGET+0.5, alpha=0.15, color='green', label='±0.5 cm')
ax1.set(ylabel='Distance (cm)', title=f'Your PID: Kp={KP}  Ki={KI}  Kd={KD}', ylim=[4, 35])
ax1.legend()

ax2.plot(t, data['P'],       color=C_P, lw=2,   label=f'P (Kp={KP})')
ax2.plot(t, data['I'],       color=C_I, lw=2,   label=f'I (Ki={KI})')
ax2.plot(t, data['D'],       color=C_D, lw=2,   label=f'D (Kd={KD})')
ax2.plot(t, data['throttle'], color='k', lw=1.5, label='Total', linestyle='--')
ax2.axhline(0, color='k', lw=0.8)
ax2.set(xlabel='Time (s)', ylabel='Term / Throttle', ylim=[-1.4, 1.4])
ax2.legend(ncol=4, fontsize=9)

plt.tight_layout()
plt.show()

# ── Score ─────────────────────────────────────────────────────────────────────
settled_idx  = np.where(np.abs(data['error']) < 0.5)[0]
settle_time  = t[settled_idx[0]] if len(settled_idx) else t[-1]
final_err    = abs(data['error'][-1])
max_over     = max(0, TARGET - min(data['distance']))
score        = 100 - (settle_time * 2) - (final_err * 15) - (max_over * 5)

print(f'Kp={KP}  Ki={KI}  Kd={KD}')
print(f'  Settle time (within 0.5 cm) : {settle_time:.1f}s')
print(f'  Final error                 : {final_err:.2f} cm')
print(f'  Max overshoot below target  : {max_over:.2f} cm')
print(f'  Score (higher = better)     : {score:.0f}')

---
## Part 8 — Connecting to Your Vehicle

Everything you've learned maps directly to the `get_pid_throttle()` function
in **vehicle-2.py**. Here's the bridge:

| Notebook | vehicle-2.py | What it is |
|----------|-------------|-----------|
| `target` | `TARGET_DISTANCE` | 10 cm goal |
| `Kp` | `KP = 0.02` | Proportional gain |
| `Ki` | `KI = 0.0001` | Integral gain |
| `Kd` | `KD = 0.002` | Derivative gain |
| `integral += error * dt` | `integral += error * dt` | Accumulate error |
| `derivative = Δerror/Δt` | `(error - last_error) / dt` | Rate of change |
| `P + I + D` | `P + I + D` → `speed` | Total output |

The vehicle runs this loop at **20 Hz** (every 0.05s), exactly like our simulation.

✏️ **PRACTICE** — Complete the full `get_pid_throttle()` function below.
This is your blueprint for what to fill into vehicle-2.py.


In [ ]:
# ✏️ PRACTICE: Complete the full PID function — mirrors vehicle-2.py exactly

import time as _time

KP              = 0.08      # proportional gain
KI              = 0.015     # integral gain
KD              = 0.5       # derivative gain
TARGET_DISTANCE = 10.0      # cm
MAX_SPEED       = 1.0
MIN_SPEED       = 0.0

# --- tracking variables (mirrors vehicle-2.py globals) ---
last_error = 0.0
integral   = 0.0
last_time  = _time.monotonic()

def get_pid_throttle(current_dist):
    global last_error, integral, last_time

    now = _time.monotonic()
    dt  = now - last_time
    if dt <= 0:
        dt = 0.001

    # ── 1. Error ──────────────────────────────────────────────────────────────
    error = ...                   # YOUR CODE: current_dist - TARGET_DISTANCE

    # ── 2. Proportional ───────────────────────────────────────────────────────
    P = ...                       # YOUR CODE: KP × error

    # ── 3. Integral ───────────────────────────────────────────────────────────
    integral += ...               # YOUR CODE: accumulate error × dt
    I = ...                       # YOUR CODE: KI × integral

    # ── 4. Derivative ─────────────────────────────────────────────────────────
    derivative = ...              # YOUR CODE: (error - last_error) / dt
    D = ...                       # YOUR CODE: KD × derivative

    # ── 5. Save state for next call ───────────────────────────────────────────
    last_error = ...              # YOUR CODE
    last_time  = ...              # YOUR CODE

    # ── 6. Combine and clamp ──────────────────────────────────────────────────
    speed = P + I + D
    return max(MIN_SPEED, min(speed, MAX_SPEED))


# ── Test your implementation ──────────────────────────────────────────────────
print('Testing your get_pid_throttle():')
last_error = 0.0; integral = 0.0; last_time = _time.monotonic()

test_distances = [30, 24, 18, 14, 12, 11, 10.5, 10.2, 10.05, 10.01]
for dist in test_distances:
    try:
        thr = get_pid_throttle(dist)
        direction = 'forward' if thr > 0.01 else ('stop' if thr < 0.01 else 'stop')
        print(f'  dist={dist:5.2f} cm → throttle={thr:+.4f}  ({direction})')
    except Exception as e:
        print(f'  Error at dist={dist}: {e}')
        break

print()
print('If these all returned sensible values → you are ready for vehicle-2.py!')

### Bonus: Tuning Intuition for the Real Vehicle

When you run your vehicle, it probably won't behave exactly like the simulation —
that's normal. Use these heuristics:

| Symptom | Likely cause | Try |
|---------|-------------|-----|
| Car inches slowly toward target, never quite arrives | Kp too small | Increase Kp |
| Car oscillates / bounces around the target | Kp too large | Decrease Kp |
| Car reaches close but always stops a bit short (with disturbance) | Ki = 0 or too small | Increase Ki |
| Adding Ki causes slow, growing oscillation | Ki too large | Decrease Ki |
| Car overshoots badly at high Kp | Kd = 0 or too small | Increase Kd |
| Car jerks / chatters nervously | Kd too large (amplifying sensor noise) | Decrease Kd |

**The golden rule:** Tune P first, then add I, then add D. Trying to tune all three
at once is very difficult.

---

## Summary

```
throttle = Kp × error              ← P: react to current error
         + Ki × integral           ← I: eliminate persistent drift
         + Kd × derivative         ← D: damp oscillation, prevent overshoot
```

P answers: **"What's wrong right now?"**
I answers: **"How long has it been wrong?"**
D answers: **"How fast is it changing?"**

Together, they give your vehicle smooth, precise, self-correcting motion.

**Now go make your vehicle never crash. 🚗🧱**
